# KAG-Pro RAG 原型验证

本 Notebook 演示完整的 RAG 管线：\n1. 加载小学数学教材文本
2. 中文语义切分
3. 向量嵌入并存入 ChromaDB
4. 提问 → 检索 → 生成答案

In [ ]:
import sys
sys.path.insert(0, '../../src')

from kag_pro.core.pipeline import RAGPipeline
from kag_pro.core.loader import DocumentLoader
from kag_pro.core.splitter import ChineseTextSplitter
from kag_pro.utils.config import get_config

config = get_config()
print(f'LLM Model: {config["llm_model"]}')
print(f'Embedding Model: {config["embedding_model"]}')
print(f'Top-K: {config["retrieval_top_k"]}')

## 步骤 1: 加载文档

In [ ]:
loader = DocumentLoader('../src/kag_pro/data/textbooks')
documents = loader.load()
print(f'加载了 {len(documents)} 个文档')
for doc in documents:
    print(f'  - {doc.metadata["source"]}: {len(doc.text)} 字')

## 步骤 2: 文本切分

In [ ]:
splitter = ChineseTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split(documents)
print(f'切分为 {len(chunks)} 个文本块')
print()
# 展示前3个块
for i, chunk in enumerate(chunks[:3]):
    meta = chunk.metadata
    print(f'--- 块 {i+1} (来源: {meta.get("source", "?")}, 字数: {len(chunk.text)}) ---')
    print(chunk.text[:150] + '...' if len(chunk.text) > 150 else chunk.text)
    print()

## 步骤 3: 初始化完整管线并索引

In [ ]:
pipeline = RAGPipeline()
indexed = pipeline.index_documents('../src/kag_pro/data/textbooks')
print(f'向量库中现有 {pipeline.document_count} 条记录')

## 步骤 4: 测试提问

In [ ]:
questions = [
    '什么是分数？',
    '1/2 和 1/3 哪个更大？为什么？',
    '三角形的内角和是多少度？',
    '如何计算长方形的面积？',
    '什么是乘法分配律？请举例说明。',
]

for q in questions:
    result = pipeline.query(q)
    print(f'❓ 问题: {q}')
    print(f'💡 回答: {result["answer"]}')
    print(f'📚 来源: {len(result["sources"])} 条')
    for s in result['sources']:
        print(f'    [{s["score"]:.2f}] {s["source"]}: {s["text"][:80]}...')
    print('---')
    print()